# One CARLA run, watched

A single parametrised trial, for **looking at the bus** rather than
measuring a campaign. `carla-campaign-towns.ipynb` answers "does the result
transfer"; this answers "what is the bus actually doing in Town03 and
Town05".

Three differences from the campaign:

1. **Every parameter is a constant** in the next cell — town, seed, policy,
   the scenario, and how CARLA renders.
2. **Nothing is recorded.** It calls `run_trial` directly, so no run file is
   written and `results/runs` is untouched. Re-running overwrites the
   in-memory results and nothing else. Findings from here are *not*
   admissible campaign evidence — reproduce them in the campaign notebook.
3. **The output is diagnostic**, not KPI-first: per-leg planned-vs-actual
   timing, every discarded route with its context, and the counters that
   separate a slow bus from a lost one.

## Before you run

* **Launch through `./jupyter-run.sh`.** `run_trial` refuses a seeded trial
  when `PYTHONHASHSEED` is unpinned, because several policies break ties
  with `min()` over a `set[str]`.
* **Bring the server up with `compose_backup.yml`.** `compose.yml`'s
  software-Vulkan variant segfaults. That file runs the server
  `-RenderOffScreen`, which is what you want with `VIEWER = True`: the
  camera still renders, so the window still works, but the server stops
  drawing a viewport nobody is looking at. Use `compose_backup_graphic.yml`
  instead if you want the server's own window as well.
* `RENDER = True` below turns the display on. It costs wall-clock time and
  changes nothing the bus decides.

## The overlay

`DEBUG_DRAW = True` draws the bus's plan into the world, so the window shows
what the tables below only describe:

* a **green line** along the route the bus was told to drive;
* a **box around every stop**, with its name over the nearest few. Amber
  where someone is waiting, blue where someone is due off, dim grey where
  the stop is idle — the counts themselves are on the request table, which
  says the same thing per passenger and covers no road;
* **counters** — simulated time, time on the current leg, the stop being
  driven to, how many passengers are waiting and aboard, and the **plan**:
  the next few stops the policy intends to visit, in order. Watching that
  list reshuffle *is* watching the policy re-plan, which no table shows;
* a **table of live requests**, top-right: where each passenger is going
  from and to, how long they have stood at the stop, how long they have been
  riding, and which of the two they are doing. Longest wait first, because
  that passenger is the closest to giving up. Fulfilled and cancelled
  requests drop off it — the table is about work outstanding;
* a **minimap**, bottom-right, with **two tabs**. *town* is the road graph
  as it lies on the ground; *by time* lays the same stops out by how long it
  takes to drive between them, so a stop a street away but a long drive sits
  far off — which is the detour factor made visible — and a stop nothing can
  reach is ringed in red. Click a tab to switch. The town view is with every stop marked, the
  plan drawn through them, and the bus on the road edge it is currently on.
  This is the graph the policy plans over rather than a picture of the town
  — a detour that looks inexplicable through the windscreen is often plain
  here, because the road the bus wanted was never an edge;
* a **permanent red marker**, labelled with the time, wherever the traffic
  manager threw the route away.

The line is what makes a dropped route legible as it happens: the bus stops
tracking it and carries on somewhere else. Markers accumulate for the whole
run, so by the end a junction that keeps doing this is a visible cluster.

⚠ All of it needs `RENDER = True`. Debug shapes are drawn by the renderer,
so with rendering off this silently shows nothing.

Three things worth knowing, all measured rather than assumed:

* The shapes are **world-space, so a building hides them**. CARLA's
  `draw_hud_*` family would have drawn through geometry, but on 0.9.16 it
  renders nothing at all.
* There is **no screen-space drawing** in CARLA at all, so `VIEWER = True`
  opens a **separate pygame window** — a chase camera with a genuine 2D HUD,
  the way CARLA's own `manual_control.py` does it. With `VIEWER = False` the
  same counters are drawn into the world instead, pinned in front of the
  spectator, which works but turns with the camera and is occluded by
  anything in the way.
* `draw_string` takes **no font size** and is a few pixels tall on a 4K
  display, so the labels are drawn as line strokes instead — see
  `src/simulation/carla_debug_text.py`. They are sized by distance, which
  holds their apparent size roughly constant.

Debug shapes expire in **simulated** time, so anything left over from a
previous run survives until the world has ticked past its lifetime.

## What to watch for

The failure modes we have evidence for, and where each shows up:

| what you see | where it lands below |
|---|---|
| bus leaves the green line | §4 route drops, `tm_next` vs `turns` |
| bus wanders, no line drawn at all | §2 `unroutable_legs` > 0, distance ratio |
| bus follows the line the long way round | §6 detour factor — structural, not a bug |
| amber bar with a big `waiting` count, bus elsewhere | §3 leg table — is it stuck on a slow leg? |
| red markers piling up in one place | §4 recurring junctions |
| bus sits still | §2 `red_light_ticks` / `queued_ticks` share |

## Configuration

In [ ]:
# ---------------------------------------------------------------- WHAT TO RUN
# The whole point of this notebook: change these, re-run, watch.
TOWN = "Town03"           # Town01, Town02, Town03, Town05
SEEDS = [0]               # one entry keeps it watchable; add more to compare
# Any of: "Fixed route", "Window W750", "Reordering W750".
# Fixed route is the §3.2.2 baseline; the other two are the contenders, L0
# and L1. More than one here runs them in turn, not side by side.
POLICY_NAMES = ["Reordering W750"]

# --------------------------------------------------------------- THE SCENARIO
# Identical to carla-campaign-towns.ipynb, so a run here is comparable to a
# campaign trial of the same town, seed and policy. Change knowingly.
NPC_VEHICLES = 20
MAX_REQUESTS = 30
DEMAND_INTERVAL_S = 90.0
CAPACITY = 20
WINDOW = 750.0            # the policy's promise
PATIENCE = 750.0          # when a waiting passenger gives up
DWELL_S = 10.0
N_STOPS = 20              # stops the policy plans ahead
COMMITMENT = 0            # locked prefix; 0 is the measured optimum
MAX_TIME = 40000.0        # ceiling; hitting it looks like a coverage failure

# ------------------------------------------------------------ CARLA / DISPLAY
RENDER = True             # False matches the campaign and runs faster
FOLLOW_CAMERA = True      # park the spectator behind the bus
# Draw the plan into the world: the route the bus was told to drive, a
# labelled bar over every stop, and a permanent red marker wherever the
# traffic manager threw that route away. Needs RENDER -- debug shapes are
# drawn by the renderer, so with RENDER = False this silently shows nothing.
DEBUG_DRAW = True
# Open a chase-camera window with a real 2D HUD, instead of reading counters
# drawn into the world. Costs about 30% of the tick rate for the camera it
# renders -- but the server no longer has to be watchable, so it can be
# launched -RenderOffScreen, which gives most of that back.
VIEWER = True
VIEWER_SIZE = (1280, 720)   # the window
# What the camera actually renders, before the frame is scaled to fit the
# window. None matches VIEWER_SIZE. Drop it below the window to buy back
# tick rate on a big display; raise it above for a sharper picture than the
# window costs to show.
RENDER_SIZE = None
MINIMAP = True            # road graph panel, bottom-right of the viewer
FIXED_DELTA_S = 0.05      # never above 0.1: past CARLA's substepping limit
REPLAN_EVERY_S = 300.0    # re-issue the turn list this often while driving
REPATH_AFTER_S = 10.0     # how long off-route before the route is rebuilt
CARLA_HOST, CARLA_PORT = "localhost", 2000
RELOAD_WORLD = False      # True forces a fresh map load before the trial

# ---------------------------------------------------------------- DIAGNOSTICS
SHOW_ALL_DROPS = False    # True prints every tagged line, not just a sample
DROP_SAMPLE = 25          # how many to show when SHOW_ALL_DROPS is False
SLOW_LEG_RATIO = 2.0      # flag a leg taking this multiple of its estimate

# What each town was calibrated against, for the section 5 check. Read from
# the topology dump, so it cannot go stale the way a copied constant does.
CALIBRATION_TOUR_RED = {"Town01": 0.321, "Town02": 0.324,
                        "Town03": 0.370, "Town05": 0.552}

In [ ]:
import re
import time
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
from cycler import cycler
from tqdm.auto import tqdm

from src.algorithm.FixedRoutePolicy import FixedRoutePolicy
from src.algorithm.ReorderingInsertionPolicy import ReorderingInsertionPolicy
from src.algorithm.WindowInsertionPolicy import WindowInsertionPolicy
from src.experiment.trial import run_trial
from src.map.CarlaMapProvider import CarlaMapProvider
from src.simulation import carla_stdout
from src.simulation.CarlaBackend import CarlaBackend, stop_positions_from
from src.simulation.carla_viewer import minimap_from, schematic_from

from IPython.display import Markdown, display

SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]
SURFACE, INK, INK_SOFT, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e6e5e1"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "axes.prop_cycle": cycler(color=SERIES),
    "axes.edgecolor": INK_SOFT, "axes.labelcolor": INK_SOFT,
    "axes.titlecolor": INK, "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.labelsize": 9, "axes.spines.top": False, "axes.spines.right": False,
    "grid.color": GRID, "grid.linewidth": 0.8, "xtick.color": INK_SOFT,
    "ytick.color": INK_SOFT, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "legend.frameon": False, "legend.fontsize": 9, "figure.dpi": 110,
})

POLICY_LABELS = ["Fixed route", "Window W750", "Reordering W750"]
POLICY_FACTORIES = {
    "Fixed route": lambda g: FixedRoutePolicy(g),
    "Window W750": lambda g: WindowInsertionPolicy(g, max_wait_s=WINDOW),
    "Reordering W750":
        lambda g: ReorderingInsertionPolicy(g, max_wait_s=WINDOW),
}

unknown = [p for p in POLICY_NAMES if p not in POLICY_FACTORIES]
if unknown:
    raise ValueError(f"unknown policies {unknown}; pick from {POLICY_LABELS}")

TRIAL = dict(
    max_requests=MAX_REQUESTS, demand_interval_s=DEMAND_INTERVAL_S,
    capacity=CAPACITY, max_wait_s=PATIENCE, dwell_s=DWELL_S,
    max_time=MAX_TIME, n_stops=N_STOPS, commitment=COMMITMENT,
)

# ---------------------------------------------------------------------------
# Collect the tagged traffic-manager messages.
#
# ``carla_stdout`` writes them straight to fd 1 with ``os.write``, precisely
# so notebook capture cannot swallow them -- which also means neither
# ``sys.stdout`` nor ``%%capture`` can read them back, and ``drain`` reports
# counts without text. ``add_listener`` is the module's way in: it hands over
# every message it attributes, tag included, and never the unrelated
# pass-through output that would otherwise pollute section 4's parse.
# Registering the same listener twice is a no-op, so re-running this cell is
# safe.
# ---------------------------------------------------------------------------
DROP_LINES: list[str] = []


def collect_drop(counter: str, line: str) -> None:
    """
    Keep one discarded-route message for section 4 to pick apart.
    :param counter: Counter name the message was recognised as. Only
        ``route_drops`` is kept, so a pattern added later cannot silently
        turn up in a table about discarded routes.
    :param line: The tagged line, exactly as it reached stdout.
    :return: None.
    """
    if counter == "route_drops":
        DROP_LINES.append(line)


carla_stdout.add_listener(collect_drop)

print(f"{TOWN}: {len(SEEDS)} seed(s) x {len(POLICY_NAMES)} policy(ies) "
      f"= {len(SEEDS) * len(POLICY_NAMES)} run(s), "
      f"rendering {'ON' if RENDER else 'off'}")

## 1. Run  🖥 *needs a live server*

Nothing here writes to `results/runs`. `run_trial` returns its result and
forgets it, so the only record is the `RUNS` dict in this kernel.

`record_animation=True` is what makes section 2 possible: it captures a
frame at every stop arrival, and the gaps between those frames are the
per-leg timings.

In [ ]:
def build_backend_factory(seed: int):
    """
    Build the CARLA backend factory for one seed.
    :param seed: Seed for NPC placement and the traffic manager.
    :return: A callable of (graph, start_stop_id) returning a CarlaBackend.
    """
    def build(graph, start_stop_id: str) -> CarlaBackend:
        """Build the CARLA backend for this trial."""
        # The trial hands the TransitGraph in here, which is the only place
        # it exists -- so this is where the cost layout gets built.
        schematic = (schematic_from(graph, MINIMAP_DATA.stops)
                     if MINIMAP_DATA is not None else None)
        return CarlaBackend(
            stop_positions=POSITIONS, position=start_stop_id, town=TOWN,
            host=CARLA_HOST, port=CARLA_PORT, fixed_delta_s=FIXED_DELTA_S,
            dwell_s=DWELL_S, npc_vehicles=NPC_VEHICLES, seed=seed,
            no_rendering=not RENDER, follow_camera=FOLLOW_CAMERA,
            debug_draw=DEBUG_DRAW, viewer=VIEWER,
            viewer_size=VIEWER_SIZE, render_size=RENDER_SIZE,
            minimap=MINIMAP_DATA,
            schematic=schematic,
            repath_after_s=REPATH_AFTER_S,
            replan_every_s=REPLAN_EVERY_S, reload_world=RELOAD_WORLD)
    return build


PROVIDER = CarlaMapProvider(TOWN)
GRAPH_LENGTHS = {e.id: e.length_m for e in PROVIDER.get_road_edges()}
POSITIONS = stop_positions_from(PROVIDER)
MINIMAP_DATA = minimap_from(PROVIDER) if MINIMAP else None

if PROVIDER.traffic_efficiency() == 1.0:
    display(Markdown(
        f"> **{TOWN} is uncalibrated** (`traffic_efficiency` 1.0). Every "
        "estimate below is free-flow, so planned-vs-actual will look far "
        "worse than it is. Calibrate it first."))

RUN_TRIALS = True    # set False to re-render the sections below only

if RUN_TRIALS:
    RUNS: dict[tuple[int, str], object] = {}
    DROPS_BY_RUN: dict[tuple[int, str], list[str]] = {}
    failures: list[tuple[int, str, str]] = []

    jobs = [(s, p) for s in SEEDS for p in POLICY_NAMES]
    bar = tqdm(jobs, desc=f"{TOWN}", unit="run")
    for seed, policy_name in bar:
        bar.set_postfix_str(f"seed {seed}, {policy_name}")
        DROP_LINES.clear()
        started = time.perf_counter()
        try:
            result = run_trial(
                policy_factory=POLICY_FACTORIES[policy_name],
                provider=PROVIDER, seed=seed,
                backend_factory=build_backend_factory(seed),
                record_animation=True, keep_tick_history=True,
                silent=True, **TRIAL)
            RUNS[(seed, policy_name)] = result
            DROPS_BY_RUN[(seed, policy_name)] = list(DROP_LINES)
        except Exception as error:          # noqa: BLE001
            # Broad on purpose: a CARLA failure surfaces as almost anything,
            # and losing the other runs of a matrix is worse than reporting.
            failures.append((seed, policy_name, repr(error)))
            tqdm.write(f"  seed {seed} {policy_name} FAILED: {error!r}")
        tqdm.write(f"  seed {seed} {policy_name}: "
                   f"{(time.perf_counter() - started) / 60:.1f} min")

    print(f"\n{len(RUNS)} run(s) held in memory, {len(failures)} failed "
          f"-- nothing written to results/runs")
else:
    print("run skipped -- set RUN_TRIALS = True with a server up")

## 2. Was the bus lost, or merely slow?

The distinction the campaign kept getting wrong. A **slow** bus drives its
planned route badly; a **lost** bus has no route at all and the traffic
manager navigates it at random, which looks identical from the outside and
inflates every distance.

The decisive number is `driven / planned`. On a healthy trial it sits near
1; the seven ruined Town05 campaign runs sat above 25.

In [ ]:
def leg_plan(origin: str, destination: str) -> tuple[float, float]:
    """
    Estimate what one leg should have cost.
    :param origin: TransitGraph node the leg starts at.
    :param destination: Node the leg ends at.
    :return: (metres, seconds) the graph predicts, (nan, nan) if unroutable.
    """
    path = GRAPH.best_path(origin, destination)
    if not path:
        return float("nan"), float("nan")
    metres = sum(GRAPH_LENGTHS.get(edge, 0.0) for edge in path)
    return metres, GRAPH.best_path_cost(origin, destination)


def legs_of(result) -> pd.DataFrame:
    """
    Turn a run's arrival frames into one row per completed leg.
    :param result: A TrialResult recorded with record_animation=True.
    :return: Frame of legs with planned and actual cost.
    """
    rows = []
    frames = result.animation_frames
    for previous, frame in zip(frames, frames[1:]):
        if frame.prev_stop_id is None:
            continue
        metres, seconds = leg_plan(frame.prev_stop_id, frame.bus_stop_id)
        actual = frame.sim_time - previous.sim_time
        rows.append({
            "from": frame.prev_stop_id, "to": frame.bus_stop_id,
            "arrived_at_s": frame.sim_time,
            "planned_s": seconds, "actual_s": actual,
            "ratio": actual / seconds if seconds else float("nan"),
            "planned_m": metres, "onboard": frame.onboard_count,
        })
    return pd.DataFrame(rows)


def tick_share(stats: dict, counter: str, sim_s: float) -> float:
    """
    Express a tick counter as a percentage of simulated time.

    Guarded because a trial that fails before its first tick reports
    ``ticks=0``, and a ZeroDivisionError here would hide the failure behind
    a traceback in the analysis.
    :param stats: The backend's counters.
    :param counter: Name of the tick counter to express.
    :param sim_s: Simulated seconds the run covered.
    :return: Percentage of simulated time, or nan for a run that never ran.
    """
    if not sim_s:
        return float("nan")
    return 100 * stats.get(counter, 0) * FIXED_DELTA_S / sim_s


GRAPH = None
summary_rows = []
for key, result in RUNS.items():
    seed, policy_name = key
    if GRAPH is None:
        GRAPH = result.transit_graph
    stats = result.backend_stats
    sim_s = stats.get("ticks", 0) * FIXED_DELTA_S
    driven_m = stats.get("distance_m", 0.0)
    legs = legs_of(result)
    planned_m = legs["planned_m"].sum() if not legs.empty else float("nan")
    summary_rows.append({
        "seed": seed, "policy": policy_name,
        "sim_s": sim_s,
        "hit_ceiling": sim_s >= MAX_TIME - 1,
        "arrivals": stats.get("arrivals", 0),
        "legs_started": stats.get("legs_started", 0),
        "driven_km": driven_m / 1000,
        "planned_km": planned_m / 1000,
        "driven/planned": driven_m / planned_m if planned_m else float("nan"),
        "unroutable": stats.get("unroutable_legs", 0),
        "route_drops": stats.get("route_drops", 0),
        "repaths": stats.get("repaths", 0),
        "replans": stats.get("replans", 0),
        "red %": tick_share(stats, "red_light_ticks", sim_s),
        "queued %": tick_share(stats, "queued_ticks", sim_s),
    })

verdicts = pd.DataFrame(summary_rows)
display(verdicts.round(2))

for row in summary_rows:
    tag = f"seed {row['seed']}, {row['policy']}"
    if not row["sim_s"]:
        display(Markdown(
            f"> ### ⚫ {tag} — never ticked\n"
            "> The backend reported no simulated time at all, so the trial "
            "died before the bus moved. Nothing below describes it; look at "
            "the server log."))
    elif row["unroutable"] > 0 or row["driven/planned"] > 3:
        display(Markdown(
            f"> ### 🔴 {tag} — the bus was LOST\n"
            f"> `{row['unroutable']:.0f}` legs had no route at all, and it "
            f"drove **{row['driven/planned']:.1f}x** the planned distance "
            f"({row['driven_km']:.1f} km against {row['planned_km']:.1f}). "
            f"When `plan_route` returns None the traffic manager keeps "
            f"driving with no destination, and nothing stops a bus that is "
            f"still moving. Treat this run's KPIs as meaningless."))
    elif row["hit_ceiling"]:
        display(Markdown(
            f"> ### 🟠 {tag} — hit the `MAX_TIME` ceiling\n"
            f"> Truncated at {MAX_TIME:.0f} s, which looks exactly like a "
            f"coverage failure. Check the leg table before reading the KPIs."))
    elif row["route_drops"] > 0:
        display(Markdown(
            f"> ### 🟡 {tag} — on route, but the turn list was discarded "
            f"{row['route_drops']:.0f} time(s)\n"
            f"> The bus recovered ({row['driven/planned']:.1f}x planned "
            f"distance), so this is the *slow* mode: each drop costs a "
            f"detour while the route is rebuilt. Section 3 says where."))
    else:
        display(Markdown(
            f"> ### 🟢 {tag} — clean\n"
            f"> No dropped routes, no unroutable legs, "
            f"{row['driven/planned']:.1f}x planned distance."))

## 3. Leg by leg

One row per stop the bus reached, with what the graph predicted against what
the drive cost. This is the table to hold next to the window: a leg you
watched go wrong has a row here, and its `ratio` says how much it cost.

A leg the bus **never** finished has no row — the arrival frame is what
creates one. A run whose rows stop early while `sim_s` runs on is the lost
bus of section 2.

In [ ]:
for key, result in RUNS.items():
    seed, policy_name = key
    display(Markdown(f"### seed {seed} — {policy_name}"))
    legs = legs_of(result)
    if legs.empty:
        display(Markdown("_The bus never reached a second stop._"))
        continue

    # Attribute each discarded route to the leg it happened during, by the
    # timestamp in its tag. The tag also names the leg, but the arrival
    # clock is what the rest of this table is indexed on.
    drop_times = [float(m.group(1)) for line in DROPS_BY_RUN.get(key, [])
                  if (m := re.search(r"\bt=([\d.]+)s", line))]
    started_at = [0.0] + list(legs["arrived_at_s"])[:-1]
    legs["drops"] = [
        sum(1 for t in drop_times if start <= t < end)
        for start, end in zip(started_at, legs["arrived_at_s"])]

    slow = legs["ratio"] > SLOW_LEG_RATIO
    display(Markdown(
        f"{len(legs)} completed legs, **{int(slow.sum())}** over "
        f"{SLOW_LEG_RATIO:g}x their estimate, "
        f"{int(legs['drops'].sum())} route drops attributed. "
        f"Median ratio **{legs['ratio'].median():.2f}x**."))
    display(legs.round(1).style.background_gradient(
        subset=["ratio"], cmap="Oranges", vmin=1, vmax=5))

    fig, ax = plt.subplots(figsize=(10, 3.2))
    ax.bar(range(len(legs)), legs["actual_s"], color=SERIES[0],
           label="actual")
    ax.plot(range(len(legs)), legs["planned_s"], color=SERIES[1],
            marker="o", ms=3, lw=1.4, label="planned")
    for i, n in enumerate(legs["drops"]):
        if n:
            ax.annotate("x" * int(n), (i, legs["actual_s"].iloc[i]),
                        ha="center", va="bottom", color=SERIES[1], fontsize=8)
    ax.set_xlabel("leg (in order driven)")
    ax.set_ylabel("seconds")
    ax.set_title(f"{TOWN} seed {seed} {policy_name} — "
                 "planned vs actual, x marks a discarded route")
    ax.grid(axis="y")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 4. Where the route was thrown away

Each of these is the traffic manager reporting it could not find the turn it
was told to make, at which point it **discards the whole remaining route**
and drives on by its own choice until something re-issues one.

The columns worth reading:

* `turns` — the list we handed it. `tm_next` — the turn it actually wanted.
  These disagreeing is the desynchronisation itself.
* `junction=True` means the message arrived while the bus was *inside* a
  junction, so the list was re-issued from a place `turns_along` cannot
  describe.
* `lane` — whether the bus had drifted off the lane its route was planned
  along.
* `issued_ago` exceeding `since_replan` by about `DWELL_S` is expected and
  is not a bug.

In [ ]:
TAG = re.compile(r"(\w+)=([^\s\]]+)")


def parse_drops(lines: list[str]) -> pd.DataFrame:
    """
    Turn tagged traffic-manager lines into a frame.
    :param lines: Raw lines captured from the backend.
    :return: One row per message, with its context fields as columns.
    """
    rows = []
    for line in lines:
        head = line.split("]", 1)[0]
        fields = dict(TAG.findall(head))
        fields["town"] = head.lstrip("[").split(" ", 1)[0]
        rows.append(fields)
    return pd.DataFrame(rows)


COLUMNS = ["t", "leg", "junction", "road", "lane", "turns", "tm_next",
           "tm_buffer", "next_junction", "issued", "since_replan", "at"]

for key in RUNS:
    seed, policy_name = key
    lines = DROPS_BY_RUN.get(key, [])
    display(Markdown(f"### seed {seed} — {policy_name}"))
    if not lines:
        display(Markdown(
            "_No routes discarded. Whatever you saw was not this._"))
        continue

    drops = parse_drops(lines)
    display(Markdown(f"**{len(drops)}** discarded routes."))

    wanted = Counter(drops.get("tm_next", pd.Series(dtype=str)).dropna())
    lane_changes = sum(v for k, v in wanted.items()
                       if k.startswith("ChangeLane"))
    display(Markdown(
        "**What the traffic manager wanted instead:** "
        + ", ".join(f"`{k}` {v} ({100 * v / len(drops):.0f}%)"
                    for k, v in wanted.most_common())
        + f"\n\nLane changes are {100 * lane_changes / len(drops):.0f}% of "
        "these. `turns_along` never emits `ChangeLane*`, so those can never "
        "match — but across the 4-town campaign they were only 7% of drops, "
        "so they are a real gap and a minority one."))

    inside = (drops.get("junction", pd.Series(dtype=str)) == "True").sum()
    display(Markdown(
        f"**Refused while inside a junction:** {inside} of {len(drops)} "
        f"({100 * inside / len(drops):.0f}%)."))

    if "next_junction" in drops:
        hotspots = Counter(drops["next_junction"].dropna()).most_common(6)
        display(Markdown(
            "**Junctions that recur** (`id@distance`): "
            + ", ".join(f"`{j}` x{n}" for j, n in hotspots if n > 1)
            or "**Junctions that recur:** none — every drop was somewhere "
               "different."))

    # A run of identical planned_from + turns is one list being refused at
    # successive junctions: the off-by-one signature, not many separate
    # faults.
    if "planned_from" in drops and "turns" in drops:
        cascades = Counter(zip(drops["planned_from"], drops["turns"]))
        worst = [(k, n) for k, n in cascades.most_common(3) if n > 1]
        if worst:
            display(Markdown(
                "**Cascades** — one issued list refused repeatedly, which is "
                "the off-by-one signature rather than separate faults: "
                + ", ".join(f"`{src}` x{n}" for (src, _), n in worst)))

    shown = drops if SHOW_ALL_DROPS else drops.head(DROP_SAMPLE)
    display(shown[[c for c in COLUMNS if c in shown.columns]])
    if not SHOW_ALL_DROPS and len(drops) > DROP_SAMPLE:
        display(Markdown(
            f"_Showing {DROP_SAMPLE} of {len(drops)}; set "
            "`SHOW_ALL_DROPS = True` for the rest._"))

## 5. Service KPIs, and whether the calibration held

The campaign numbers, for the same run you just watched. Read the fulfilment
rate first: wait and ride are means over *fulfilled* requests only, so a bus
that abandons its hard riders posts better numbers by giving up.

The speed check asks whether `traffic_efficiency` — fitted on a fixed tour
of every stop — describes the legs a policy actually chooses. Across the
four campaign towns the run came out 8–17% slower than its tour, every time.

In [ ]:
rows = []
for key, result in RUNS.items():
    seed, policy_name = key
    kpi, stats = result.kpi, result.backend_stats
    sim_s = stats.get("ticks", 0) * FIXED_DELTA_S
    waits = result.wait_times
    rows.append({
        "seed": seed, "policy": policy_name,
        "served %": 100 * kpi.served_fraction,
        "fulfilled": len(waits),
        "mean wait s": kpi.mean_wait_time,
        "mean ride s": kpi.mean_ride_time,
        "throughput/min": kpi.throughput,
        "breach %": (100 * sum(w > WINDOW for w in waits) / len(waits)
                     if waits else float("nan")),
        "speed m/s": stats.get("distance_m", 0.0) / sim_s if sim_s else 0.0,
        "red %": tick_share(stats, "red_light_ticks", sim_s),
    })

kpis = pd.DataFrame(rows)
display(kpis.round(2))

efficiency = PROVIDER.traffic_efficiency()
tour_red = CALIBRATION_TOUR_RED.get(TOWN)
notes = [
    f"`traffic_efficiency` in use for **{TOWN}**: **{efficiency:.4f}** "
    "(read from the topology dump, so it cannot be a stale copy).",
]
if tour_red is not None:
    observed_red = kpis["red %"].mean()
    notes.append(
        f"Red-light share: **{observed_red:.1f}%** here against "
        f"**{100 * tour_red:.1f}%** on the calibration tour "
        f"({observed_red - 100 * tour_red:+.1f} pp). Red share transfers "
        "well between the tour and a campaign; overall speed does not "
        "quite.")
if WINDOW >= PATIENCE:
    notes.append(
        f"`window == patience == {WINDOW:.0f} s`, so **breach is zero by "
        "construction** — anyone who waits that long is already cancelled. "
        "Anything but 0.00 is a bug, not a finding.")
display(Markdown("\n\n".join(notes)))

## 6. What this town costs before the bus turns a wheel

Structural, measured off the graph alone — no server needed, and true of
every run in this town. The **detour factor** is the graph's shortest path
between two stops divided by the straight line between them.

Measured 2026-08-15 across the campaign towns:

| town | edges | detour | median leg | est. time | efficiency |
|---|---|---|---|---|---|
| Town01 | 139 | 2.81 | 699 m | 140 s | 0.455 |
| Town02 | 103 | 2.40 | 366 m | 89 s | 0.410 |
| Town03 | 423 | **3.50** | 889 m | **250 s** | 0.365 |
| Town05 | 497 | **4.10** | 706 m | **277 s** | 0.271 |

All four towns have a **campus-sized stop footprint** — 200–413 m across, 15
stops. Town03 and Town05 are not bigger; they are more expensive per metre,
because `TransitGraph` has no lane-change edges and a multi-lane town's
graph therefore forces long way-arounds. A long way round that you watch and
think is a bug is very often this instead.

In [ ]:
import itertools
import math
import statistics

stops = PROVIDER.get_stops()
coord = {s.edge.id: (s.x, s.y) for s in stops}
ids = [s.edge.id for s in stops]

detours, times, unreachable = [], [], 0
for a, b in itertools.permutations(ids, 2):
    path = GRAPH.best_path(a, b)
    if not path:
        unreachable += 1
        continue
    metres = sum(GRAPH_LENGTHS.get(e, 0.0) for e in path)
    straight = math.dist(coord[a], coord[b])
    if straight > 20 and metres > 0:
        detours.append(metres / straight)
        times.append(GRAPH.best_path_cost(a, b))

pairs = len(ids) * (len(ids) - 1)
xs = [s.x for s in stops]
ys = [s.y for s in stops]
display(Markdown(
    f"**{TOWN}** — {len(stops)} stops, {len(PROVIDER.get_road_edges())} road "
    f"edges, `traffic_efficiency` {PROVIDER.traffic_efficiency():.4f}\n\n"
    f"* stop footprint: **{max(max(xs) - min(xs), max(ys) - min(ys)):.0f} m** "
    "across\n"
    f"* median detour factor: **{statistics.median(detours):.2f}x**\n"
    f"* median estimated leg: **{statistics.median(times):.0f} s**\n"
    f"* unreachable stop pairs: **{unreachable} of {pairs}** "
    f"({100 * unreachable / pairs:.0f}%)"))

# Otherwise-silent map faults: a stop that matched no segment, an edge
# naming a node that does not exist, a node the bus can never leave. Town03
# reports 9 of these and Town05 4, and a stop the bus cannot reach explains
# a leg you watched it refuse to drive.
problems = PROVIDER.validate()
if problems:
    display(Markdown(
        f"**{len(problems)} map problems in {TOWN}:**\n\n"
        + "\n".join(f"* `{p}`" for p in problems[:12])
        + ("\n* …" if len(problems) > 12 else "")))
else:
    display(Markdown(f"**No map problems reported for {TOWN}.**"))

## 7. Demand over the run

Pending and onboard counts per tick. Pending climbing and never coming down
is the bus losing the race; a sawtooth is it keeping up. A flat line after
some point means demand stopped (`MAX_REQUESTS` reached) and the tail is the
bus working through the backlog — or failing to.

In [ ]:
for key, result in RUNS.items():
    seed, policy_name = key
    history = result.tick_history
    if not history:
        display(Markdown(
            f"_seed {seed} {policy_name}: no tick history retained._"))
        continue
    t = [s.sim_time / 60 for s in history]
    fig, ax = plt.subplots(figsize=(10, 3.2))
    ax.plot(t, [s.pending_count for s in history], color=SERIES[0],
            lw=1.4, label="pending")
    ax.plot(t, [s.occupancy for s in history], color=SERIES[2],
            lw=1.4, label="onboard")
    ax.set_xlabel("simulated minutes")
    ax.set_ylabel("requests")
    ax.set_title(f"{TOWN} seed {seed} {policy_name} — demand over the run")
    ax.grid(axis="y")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 8. Reading this against what you saw

A rough decision procedure, in the order the evidence should be consulted:

1. **§2 says lost** (`unroutable > 0`, or `driven/planned` above 3). Then
   the bus genuinely had no destination and everything downstream is noise.
   What you watched was CARLA's traffic manager, not our policy.
2. **§2 says clean but §4 lists drops.** The bus was on route and lost it at
   specific junctions — which is the moment you saw it leave the green line.
   Every marker is labelled with its `t=`, and the same `t=` appears in §4's
   table and in the `drops` column of §3's leg table, so a moment in the
   window, a marker on the map and a row in a table are the same event seen
   three ways.
3. **No drops, but §3 shows a high `ratio`.** The route was kept and driving
   it simply cost more than the estimate — traffic, lights, or a
   `traffic_efficiency` that does not describe this leg. Town03's
   between-leg sigma is 0.9468 against Town01's 0.278, so one factor
   describes its legs poorly.
4. **Everything looks fine and the coverage is still bad.** Then it is the
   scenario, not the driving: 30 requests at one per 90 s with a 750 s
   window is more than one bus can serve where a median leg costs 250–277 s.

What would change our minds about the campaign: a Town03 or Town05 run that
is clean by §2, low-ratio by §3, and *still* fails to serve — that would
move the fault from the map layer to the policy.